# ⚡ Matriz Energética Argentina — ETL (v2.1)

**Fuentes:** CAMMESA (demanda y potencia instalada), Secretaría de Energía (SESCO: producción de petróleo y gas por cuenca; producción por empresa), ENARGAS (consumo de gas por sector) e INDEC (comercio exterior energético), vía datos.gob.ar y datos.energia.gob.ar.

**Novedades v2.1:** si SESCO no responde, la producción de petróleo se empalma: total país oficial hasta nov-2025 y luego la suma por empresa ajustada por nivel (las principales empresas cubren ~70% del total).

**Novedades v2:**
- Producción de petróleo con fuente **vigente** (SESCO por cuenca; el dataset 363 que usaba la v1 quedó congelado en nov-2025). Con doble respaldo: suma por empresa (dataset 365) y el dataset viejo.
- Nueva hoja **cuencas** (producción de petróleo y gas por cuenca, para mapa/aperturas en Power BI).
- Nueva hoja **cobertura** con *alerta de atraso* por serie: si una fuente deja de actualizarse (como pasó con petróleo), queda marcado en el tablero en vez de pasar inadvertido.
- Guardado compatible con Colab **y** con ejecución automática (service account) para la futura actualización programada.
- Celda final de *diagnóstico* que busca series de generación por fuente y precios del MEM para la próxima iteración.

**Salida:** planilla *Energía AR - Datos limpios*. **Para actualizar:** *Ejecutar todas* y después *Actualizar* en Power BI.


In [ ]:
# =============================================================
# 1) CONFIGURACIÓN Y HELPERS - MATRIZ ENERGÉTICA ARGENTINA (v2)
# =============================================================
import requests, time, io, unicodedata
import pandas as pd, numpy as np

SPREADSHEET_ID = "192EzFh_qcs87tPY72S8YgqXy1Sq6bSfouhz6taw7-kw"   # "Energía AR - Datos limpios" en tu Drive
DESDE = "2016-01-01"
CSV = "https://infra.datos.gob.ar/catalog/sspm/dataset"
API_SERIES = "https://apis.datos.gob.ar/series/api/series/"
CKAN_ENERGIA = "http://datos.energia.gob.ar/api/3/action"

errores = []          # (bloque, mensaje)
fuentes_usadas = []   # (bloque, fuente elegida) -> queda registrado en la hoja "info"

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

def leer(ruta):   # CSV del catálogo sspm (infra.datos.gob.ar)
    r = requests.get(f"{CSV}/{ruta}", headers=UA, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), parse_dates=["indice_tiempo"]).set_index("indice_tiempo").sort_index()

def leer_url(url, **kw):   # CSV genérico desde cualquier URL
    r = requests.get(url, headers=UA, timeout=180)
    r.raise_for_status()
    return pd.read_csv(io.BytesIO(r.content), **kw)

def serie_api(sid):
    r = requests.get(API_SERIES, params={"ids": sid, "start_date": DESDE, "limit": 5000, "format": "json"},
                     headers=UA, timeout=60).json()
    return pd.Series({pd.Timestamp(f): v for f, v in r["data"] if v is not None}, dtype=float)

def _norm(s):   # minúsculas y sin tildes, para comparar nombres de recursos
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()

def ckan_recurso(dataset_ids, *palabras, formato="CSV"):
    """Busca en datos.energia.gob.ar un recurso cuyo nombre contenga todas las palabras.
    dataset_ids: lista de slugs/UUIDs posibles del dataset. Devuelve (nombre, url) o lanza error."""
    ultimo_error = None
    for did in dataset_ids:
        try:
            j = requests.get(f"{CKAN_ENERGIA}/package_show", params={"id": did}, headers=UA, timeout=(10, 60)).json()
            if not j.get("success"):
                continue
            for res in j["result"]["resources"]:
                nombre = _norm(res.get("name", ""))
                if res.get("format", "").upper() == formato and all(_norm(p) in nombre for p in palabras):
                    return res.get("name"), res.get("url")
        except Exception as e:
            ultimo_error = e
    raise RuntimeError(f"recurso no encontrado ({' / '.join(map(str, palabras))}): {ultimo_error}")

def detectar_fecha(df):
    """Devuelve el df indexado por la columna de fecha que encuentre."""
    for c in df.columns:
        cn = _norm(c)
        if cn in ("indice_tiempo", "fecha", "fecha_data", "periodo", "mes") or "fecha" in cn:
            s = pd.to_datetime(df[c], errors="coerce")
            if s.notna().mean() > 0.8:
                return df.assign(**{c: s}).dropna(subset=[c]).set_index(c).sort_index()
    # formato año + mes en columnas separadas
    cols = {_norm(c): c for c in df.columns}
    if ("anio" in cols or "ano" in cols or "año" in cols) and "mes" in cols:
        ca = cols.get("anio") or cols.get("ano") or cols.get("año")
        f = pd.to_datetime(dict(year=df[ca], month=df[cols["mes"]], day=1), errors="coerce")
        return df.assign(_f=f).dropna(subset=["_f"]).set_index("_f").sort_index()
    raise RuntimeError(f"no encontré columna de fecha entre {list(df.columns)[:12]}")

print("Configuración lista")


## Descarga

In [ ]:
# =============================================================
# 2) DESCARGA
# =============================================================
fuentes = {}

# --- 2a. Catálogo sspm: demanda eléctrica, gas, refinación/ventas (v1) y empresas ---
for nombre, ruta in {
    "electricidad_m": "367/distribution/367.3/download/demanda-de-electricidad-datos-mensuales.csv",
    "electricidad_a": "367/distribution/367.1/download/demanda-de-electricidad-datos-anuales.csv",
    "gas":            "364/distribution/364.3/download/actividad-gas.csv",
    "petroleo_v1":    "363/distribution/363.3/download/exportaciones-actividad-saldo-comercial-rangos-exportacion-empresa-exportadora-mensual.csv",
    "empresas":       "365/distribution/365.3/download/actividad-empresas.csv",
}.items():
    try:
        fuentes[nombre] = leer(ruta)
        print(f"✔ {nombre:15s} hasta {fuentes[nombre].index.max().date()}  ({fuentes[nombre].shape[1]} columnas)")
    except Exception as e:
        errores.append((nombre, str(e)[:150])); print(f"✘ {nombre}: {e}")

# --- 2b. INDEC: comercio exterior energético (API Series de Tiempo) ---
for nombre, sid in {"expo_energia": "74.3_IECE_0_M_35", "impo_combustibles": "74.3_IICL_0_M_42"}.items():
    try:
        fuentes[nombre] = serie_api(sid); print(f"✔ {nombre:15s} hasta {fuentes[nombre].index.max().date()}")
    except Exception as e:
        errores.append((nombre, str(e)[:150])); print(f"✘ {nombre}: {e}")

# --- 2c. SESCO (datos.energia.gob.ar): producción de petróleo y gas por cuenca (fuente VIGENTE) ---
SESCO_IDS = ["produccion-de-petroleo-y-gas-sesco", "590d1284-fd6d-4686-afd8-b3da5d90a6e9"]

def sesco_por_cuenca(*palabras):
    nombre, url = ckan_recurso(SESCO_IDS, *palabras)
    df = leer_url(url)
    df = detectar_fecha(df)
    cols = {_norm(c): c for c in df.columns}
    # formato largo (fecha, cuenca, valor) o ancho (una columna por cuenca)
    col_cuenca = next((cols[c] for c in cols if "cuenca" in c), None)
    if col_cuenca is not None:
        col_val = next(c for c in df.columns if c != col_cuenca and pd.api.types.is_numeric_dtype(df[c]))
        ancho = df.pivot_table(index=df.index, columns=col_cuenca, values=col_val, aggfunc="sum")
    else:
        ancho = df.select_dtypes("number")
    ancho = ancho[ancho.index >= DESDE]
    return nombre, ancho   # columnas = cuencas, valores = promedio diario

cuencas_petro = cuencas_gas = None
try:
    nom, cuencas_petro = sesco_por_cuenca("petroleo", "promedio", "cuenca")
    fuentes_usadas.append(("petroleo_produccion", f"SESCO: {nom}"))
    tot = cuencas_petro.sum(axis=1)
    print(f"✔ SESCO petróleo por cuenca hasta {cuencas_petro.index.max().date()} | total último mes ≈ {tot.iloc[-1]:,.0f} m³/d")
except Exception as e:
    errores.append(("sesco_petroleo", str(e)[:150])); print(f"✘ SESCO petróleo: {e}")
try:
    nom, cuencas_gas = sesco_por_cuenca("gas", "promedio", "cuenca")
    fuentes_usadas.append(("gas_cuencas", f"SESCO: {nom}"))
    print(f"✔ SESCO gas por cuenca hasta {cuencas_gas.index.max().date()} | total último mes ≈ {cuencas_gas.sum(axis=1).iloc[-1]:,.0f} miles de m³/d")
except Exception as e:
    errores.append(("sesco_gas", str(e)[:150])); print(f"✘ SESCO gas: {e}")

# --- 2d. Ventas de combustibles al mercado (candidatos; si fallan, queda la fuente v1 con alerta) ---
VENTAS_IDS = ["ventas-de-combustibles-lquidos-al-mercado", "ventas-de-combustibles-liquidos-al-mercado",
              "ventas-de-combustibles-liquidos", "ventas-excluyendo-ventas-a-empresas-del-sector"]
ventas_nuevas = None
try:
    nom, url = ckan_recurso(VENTAS_IDS, "venta")
    df = detectar_fecha(leer_url(url))
    cols = {_norm(c): c for c in df.columns}
    col_prod = next((cols[c] for c in cols if c in ("producto", "combustible") or "producto" in c), None)
    if col_prod is None:
        raise RuntimeError(f"sin columna de producto; columnas: {list(df.columns)[:12]}")
    col_val = next(c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]))
    p = _norm_series = df[col_prod].map(_norm)
    df["_grupo"] = np.where(p.str.contains("gasoil|gas oil"), "gasoil",
                   np.where(p.str.contains("nafta"), "naftas", None))
    ventas_nuevas = (df.dropna(subset=["_grupo"]).groupby([pd.Grouper(freq="MS"), "_grupo"])[col_val]
                       .sum().unstack("_grupo") / 1000)     # m³ -> miles de m³ (verificar escala en el print)
    ventas_nuevas = ventas_nuevas[ventas_nuevas.index >= DESDE]
    fuentes_usadas.append(("ventas_combustibles", f"datos.energia.gob.ar: {nom}"))
    print(f"✔ ventas combustibles ({nom}) hasta {ventas_nuevas.index.max().date()} | último mes: "
          + ", ".join(f"{c} ≈ {v:,.0f}" for c, v in ventas_nuevas.iloc[-1].items()))
except Exception as e:
    errores.append(("ventas_nuevas", str(e)[:150])); print(f"✘ ventas combustibles (candidatos nuevos): {e}")

print("\n⚠ Revisar escalas impresas arriba: petróleo total país ≈ 100.000-200.000 m³/d; gas ≈ 130.000-160.000 miles de m³/d;")
print("  gasoil ≈ 1.100-1.600 miles de m³/mes; naftas ≈ 700-1.000 miles de m³/mes. Si difiere en x1000, avisarme y ajustamos la conversión.")


## Limpieza y cálculos

In [ ]:
# =============================================================
# 3) LIMPIEZA Y CÁLCULOS
# =============================================================
# (id, nombre, grupo, unidad, fuente, polaridad, orden, origen, columna)
IND = [
    ("demanda_total",        "Demanda eléctrica total",          "Electricidad", "GWh",  "CAMMESA", 0, 1,  "electricidad_m", "demanda_total"),
    ("demanda_residencial",  "Demanda eléctrica residencial",    "Electricidad", "GWh",  "CAMMESA", 0, 2,  "electricidad_m", "demanda_residencial"),
    ("demanda_comercio_ind", "Demanda comercio e industria",     "Electricidad", "GWh",  "CAMMESA", 0, 3,  "electricidad_m", "comercio_e_industria"),
    ("demanda_grandes",      "Demanda grandes usuarios",         "Electricidad", "GWh",  "CAMMESA", 0, 4,  "electricidad_m", "grandes_usuarios"),
    ("potencia_maxima",      "Potencia máxima demandada",        "Electricidad", "MW",   "CAMMESA", 0, 5,  "electricidad_m", "potencia_maxima"),
    ("temperatura",          "Temperatura promedio",             "Electricidad", "°C",   "CAMMESA", 0, 6,  "electricidad_m", "temperatura_promedio"),
    ("gas_produccion",       "Producción de gas natural",        "Gas natural",  "millones de m³", "Secretaría de Energía", 1, 10, "gas", "produccion_gas_natural"),
    ("gas_consumo_total",    "Consumo de gas natural",           "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 11, "gas", "total"),
    ("gas_residencial",      "Gas: residencial",                 "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 12, "gas", "residencial"),
    ("gas_industria",        "Gas: industria",                   "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 13, "gas", "industria"),
    ("gas_centrales",        "Gas: centrales eléctricas",        "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 14, "gas", "centrales_electricas"),
    ("gas_gnc",              "Gas: GNC",                         "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 15, "gas", "gnc"),
    ("gas_comercial",        "Gas: comercial y entes oficiales", "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 16, "gas", ["comercial", "entes_oficiales"]),
    ("gas_sdb",              "Gas: subdistribuidoras (SDB)",     "Gas natural",  "millones de m³", "ENARGAS / Secretaría de Energía", 0, 17, "gas", "sdb"),
    ("petroleo_procesado",   "Petróleo procesado en refinerías", "Petróleo",     "miles de m³", "Secretaría de Energía", 0, 21, "petroleo_v1", "petroleo_crudo_procesado"),
    ("gasoil_ventas",        "Ventas de gasoil",                 "Petróleo",     "miles de m³", "Secretaría de Energía", 0, 22, "petroleo_v1", "ventas_gasoil"),
    ("naftas_ventas",        "Ventas de naftas",                 "Petróleo",     "miles de m³", "Secretaría de Energía", 0, 23, "petroleo_v1", ["ventas_nafta_comun_mayor83_ron", "ventas_nafta_super_mayor93_ron", "ventas_nafta_mayor97_ron"]),
    ("expo_energia",         "Exportaciones de combustibles y energía", "Comercio exterior", "millones de USD", "INDEC", 1, 30, "expo_energia", None),
    ("impo_combustibles",    "Importaciones de combustibles y lubricantes", "Comercio exterior", "millones de USD", "INDEC", -1, 31, "impo_combustibles", None),
]
filas = []
for (iid, nom, grupo, uni, fue, pol, orden, origen, colname) in IND:
    if origen not in fuentes: continue
    src = fuentes[origen]
    if colname is None: s_ = src
    elif isinstance(colname, list): s_ = src[colname].sum(axis=1, min_count=1)
    else: s_ = src[colname]
    s_ = s_[s_.index >= DESDE].dropna()
    filas += [(f, iid, float(v)) for f, v in s_.items()]
ser = pd.DataFrame(filas, columns=["fecha", "id", "valor"])

# --- Producción de petróleo: SESCO (vigente) -> empalme 363 + empresas (365) -> solo 363 ---
petro_total_m3d = None
fuente_petro = None
v1_m3d = emp_m3d = None
if "petroleo_v1" in fuentes and "produccion_petroleo_crudo" in fuentes["petroleo_v1"]:
    m = fuentes["petroleo_v1"]["produccion_petroleo_crudo"].dropna()    # miles de m³/mes, total país (congelado nov-2025)
    v1_m3d = m * 1000 / m.index.days_in_month
if "empresas" in fuentes:
    e = fuentes["empresas"]
    cols = [c for c in e.columns if c.startswith("produccion_petroleo_crudo_")]
    if cols:
        mensual_m3 = e[cols].sum(axis=1, min_count=1).dropna()          # m³/mes, solo principales empresas (parcial)
        emp_m3d = mensual_m3 / mensual_m3.index.days_in_month

if cuencas_petro is not None and len(cuencas_petro):
    petro_total_m3d = cuencas_petro.sum(axis=1)
    fuente_petro = "Secretaría de Energía (SESCO)"
elif v1_m3d is not None and emp_m3d is not None:
    # Empalme: total país oficial hasta su último mes y, después, la suma por empresa
    # ajustada por nivel (las empresas listadas no cubren el 100% de la producción).
    comun = v1_m3d.index.intersection(emp_m3d.index)[-12:]
    factor = float((v1_m3d[comun] / emp_m3d[comun]).median())
    extension = emp_m3d[emp_m3d.index > v1_m3d.index.max()] * factor
    petro_total_m3d = pd.concat([v1_m3d, extension]).sort_index()
    fuente_petro = (f"Secretaría de Energía (total país hasta {v1_m3d.index.max():%m/%Y}; "
                    f"luego extendido con producción por empresa x{factor:.2f})")
    fuentes_usadas.append(("petroleo_produccion", f"empalme 363 + empresas (factor {factor:.3f})"))
    print(f"✔ petróleo empalmado: factor de cobertura {factor:.3f}, extendido hasta {petro_total_m3d.index.max().date()}")
elif v1_m3d is not None:
    petro_total_m3d = v1_m3d
    fuente_petro = "Secretaría de Energía (dataset 363, DESACTUALIZADO)"
    fuentes_usadas.append(("petroleo_produccion", "solo dataset 363 (desactualizado)"))
elif emp_m3d is not None:
    petro_total_m3d = emp_m3d
    fuente_petro = "Secretaría de Energía (suma por empresa; PARCIAL)"
    fuentes_usadas.append(("petroleo_produccion", "solo suma por empresa (parcial)"))

if petro_total_m3d is not None:
    petro_total_m3d = petro_total_m3d[petro_total_m3d.index >= DESDE].dropna()
    prod_mensual = (petro_total_m3d * petro_total_m3d.index.days_in_month / 1000)  # miles de m³/mes
    ser = pd.concat([ser,
        pd.DataFrame({"fecha": prod_mensual.index, "id": "petroleo_produccion", "valor": prod_mensual.values}),
        pd.DataFrame({"fecha": petro_total_m3d.index, "id": "petroleo_bpd",
                      "valor": (petro_total_m3d * 6.2898 / 1000).values})])        # miles de barriles/día
IND += [("petroleo_produccion", "Producción de petróleo crudo", "Petróleo", "miles de m³", fuente_petro or "s/d", 1, 20, None, None),
        ("petroleo_bpd", "Producción de petróleo", "Petróleo", "miles de barriles/día", fuente_petro or "s/d", 1, 19, None, None)]

# --- Ventas de combustibles: si la fuente nueva anda, pisa las series v1 (congeladas) ---
if ventas_nuevas is not None and len(ventas_nuevas):
    ser = ser[~ser["id"].isin(["gasoil_ventas", "naftas_ventas"])]
    for g, iid in [("gasoil", "gasoil_ventas"), ("naftas", "naftas_ventas")]:
        if g in ventas_nuevas:
            s_ = ventas_nuevas[g].dropna()
            ser = pd.concat([ser, pd.DataFrame({"fecha": s_.index, "id": iid, "valor": s_.values})])

# --- Derivados: saldo comercial energético y gas en MMm³/día ---
piv = ser.pivot_table(index="fecha", columns="id", values="valor")
dias = piv.index.days_in_month
extra = []
if {"expo_energia", "impo_combustibles"} <= set(piv):
    extra.append(("saldo_energetico", piv["expo_energia"] - piv["impo_combustibles"]))
if "gas_produccion" in piv:
    extra.append(("gas_mm3d", piv["gas_produccion"] / dias))
for iid, s_ in extra:
    ser = pd.concat([ser, pd.DataFrame({"fecha": s_.dropna().index, "id": iid, "valor": s_.dropna().values})])
IND += [("saldo_energetico", "Saldo comercial energético", "Comercio exterior", "millones de USD", "INDEC (cálculo propio)", 1, 32, None, None),
        ("gas_mm3d", "Producción de gas natural", "Gas natural", "millones de m³/día", "Secretaría de Energía (cálculo propio)", 1, 9, None, None)]
indicadores = pd.DataFrame([r[:7] for r in IND], columns=["id", "nombre", "grupo", "unidad", "fuente", "polaridad", "orden"])
indicadores = indicadores[indicadores.id.isin(ser.id.unique())].reset_index(drop=True)

ser = ser.sort_values(["id", "fecha"]).reset_index(drop=True)
g = ser.groupby("id")["valor"]
ser["var_interanual"] = np.where(ser["id"].isin(["saldo_energetico", "temperatura"]), g.diff(12), g.pct_change(12, fill_method=None) * 100)
ser["suma_12m"] = g.transform(lambda s_: s_.rolling(12, min_periods=12).sum())
ser["suma_anio"] = ser.groupby(["id", ser["fecha"].dt.year])["valor"].cumsum()
ser = ser.replace([np.inf, -np.inf], np.nan)

# --- Hoja cuencas (para mapa / apertura por cuenca en Power BI) ---
cu = []
for prod, w, unidad in [("Petróleo", cuencas_petro, "m³/día"), ("Gas natural", cuencas_gas, "miles de m³/día")]:
    if w is None: continue
    for c in w.columns:
        s_ = w[c].dropna()
        cu += [(f, prod, str(c).strip().title(), float(v), unidad) for f, v in s_.items()]
cuencas = pd.DataFrame(cu, columns=["fecha", "producto", "cuenca", "valor", "unidad"])
if len(cuencas):
    cuencas["var_interanual"] = cuencas.sort_values("fecha").groupby(["producto", "cuenca"])["valor"].pct_change(12, fill_method=None) * 100

# --- Potencia instalada por tecnología (igual que v1) ---
TEC = {"potencia_instalada_ciclos_combinados": ("Ciclos combinados", "Térmica fósil"),
       "potencia_instalada_turbina_a_gas": ("Turbinas a gas", "Térmica fósil"),
       "potencia_instalada_turbovapor": ("Turbovapor", "Térmica fósil"),
       "potencia_instalada_motor_diesel": ("Motores diésel", "Térmica fósil"),
       "potencia_instalada_hidraulica": ("Hidráulica (grandes represas)", "Hidráulica"),
       "potencia_instalada_nuclear": ("Nuclear", "Nuclear"),
       "potencia_instalada_eolica": ("Eólica", "Renovable"),
       "potencia_instalada_solar": ("Solar", "Renovable"),
       "potencia_instalada_biogas": ("Biogás", "Renovable"),
       "potencia_instalada_hidraulica_renovable": ("Pequeños aprovechamientos hidráulicos", "Renovable")}
pot = []
if "electricidad_a" in fuentes:
    ea = fuentes["electricidad_a"]
    for c, (tec, grupo) in TEC.items():
        if c in ea: pot += [(f.year, tec, grupo, float(v)) for f, v in ea[c].dropna().items()]
potencia = pd.DataFrame(pot, columns=["anio", "tecnologia", "grupo", "mw"])
if len(potencia):
    potencia["participacion"] = potencia["mw"] / potencia.groupby("anio")["mw"].transform("sum") * 100

# --- Producción por empresa (igual que v1) ---
emp = []
if "empresas" in fuentes:
    e = fuentes["empresas"]; e = e[e.index >= DESDE]
    NOMBRES = {"ypf_sa": "YPF", "pan_american_energy_llc": "Pan American Energy", "pluspetrol_sa": "Pluspetrol",
               "sinopec_argentina": "Sinopec", "sinopec_argentina_exploration": "Sinopec", "tecpetrol_sa": "Tecpetrol",
               "total_austral_sa": "TotalEnergies", "chevron_argentina_srl": "Chevron", "enap_sipetrol_argentina_sa": "ENAP Sipetrol",
               "petrobras_argentina_sa": "Petrobras"}
    for c in e.columns:
        for pref, prod in [("produccion_petroleo_crudo_", "Petróleo"), ("produccion_gas_natural_", "Gas natural")]:
            if c.startswith(pref):
                nom = NOMBRES.get(c[len(pref):], c[len(pref):])
                emp += [(f, nom, prod, float(v) / 1000) for f, v in e[c].dropna().items() if v > 0]
empresas = pd.DataFrame(emp, columns=["fecha", "empresa", "producto", "valor"])

print(ser.groupby("id")["fecha"].agg(["min", "max", "count"]).to_string())
if len(cuencas): print("\nCuencas:", cuencas.cuenca.nunique(), "| hasta", cuencas.fecha.max().date())


## KPIs, cobertura y guardado

In [ ]:
# =============================================================
# 4) KPIs, COBERTURA CON ALERTAS, RESUMEN Y GUARDADO
# =============================================================
MESES = ["enero","febrero","marzo","abril","mayo","junio","julio","agosto","septiembre","octubre","noviembre","diciembre"]
mes = lambda f: f"{MESES[f.month-1]} {f.year}"
n = lambda x, d=1: "s/d" if x is None or pd.isna(x) else f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def ult(i):
    g = ser[(ser.id == i) & ser.valor.notna()]
    return g.iloc[-1] if len(g) else None

kpis = pd.DataFrame([dict(id=i, **ult(i)[["fecha", "valor", "var_interanual", "suma_12m"]].to_dict()) for i in indicadores.id if ult(i) is not None])

# --- Cobertura y alerta de atraso (meses sin dato nuevo vs. el rezago normal de cada fuente) ---
hoy_m = pd.Timestamp.today().normalize().replace(day=1)
REZAGO_OK = {"expo_energia": 3, "impo_combustibles": 3, "saldo_energetico": 3}   # rezago aceptable por serie (default 4 meses)
cob = ser.dropna(subset=["valor"]).groupby("id")["fecha"].agg(desde="min", hasta="max", observaciones="count").reset_index()
cob["meses_atraso"] = ((hoy_m.year - cob.hasta.dt.year) * 12 + hoy_m.month - cob.hasta.dt.month)
cob["alerta"] = np.where(cob.apply(lambda r: r.meses_atraso > REZAGO_OK.get(r.id, 4), axis=1), "⚠ fuente atrasada", "ok")
cobertura = cob.merge(indicadores[["id", "nombre", "grupo", "fuente", "orden"]], on="id").sort_values("orden")
atrasadas = cobertura[cobertura.alerta != "ok"]
if len(atrasadas):
    print("⚠ SERIES ATRASADAS:\n" + atrasadas[["id", "hasta", "meses_atraso", "fuente"]].to_string(index=False) + "\n")

textos = []
p, gp, d, s_ = ult("petroleo_bpd"), ult("gas_mm3d"), ult("demanda_total"), ult("saldo_energetico")
if p is not None and gp is not None:
    textos.append(("Hidrocarburos", f"La producción de petróleo fue de {n(p.valor,0)} mil barriles diarios en {mes(p.fecha)} ({n(p.var_interanual)}% interanual) y la de gas de {n(gp.valor)} millones de m³ diarios en {mes(gp.fecha)} ({n(gp.var_interanual)}% i.a.)."))
if d is not None:  textos.append(("Electricidad", f"La demanda eléctrica de {mes(d.fecha)} fue de {n(d.valor,0)} GWh ({n(d.var_interanual)}% interanual); en los últimos 12 meses sumó {n(d.suma_12m,0)} GWh."))
if s_ is not None: textos.append(("Comercio exterior", f"La balanza comercial energética dejó USD {n(s_.valor,0)} millones en {mes(s_.fecha)} y acumula USD {n(s_.suma_12m,0)} millones en 12 meses ({'superávit' if s_.suma_12m > 0 else 'déficit'})."))
if len(potencia):
    a = potencia.anio.max(); pa = potencia[potencia.anio == a].groupby("grupo")["participacion"].sum()
    textos.append(("Matriz eléctrica", f"En {a} la potencia instalada fue de {n(potencia[potencia.anio == a].mw.sum(),0)} MW: térmica fósil {n(pa.get('Térmica fósil',0))}%, hidráulica {n(pa.get('Hidráulica',0))}%, renovables {n(pa.get('Renovable',0))}% y nuclear {n(pa.get('Nuclear',0))}%."))
if len(cuencas):
    ult_f = cuencas[cuencas.producto == "Petróleo"].fecha.max()
    cp = cuencas[(cuencas.producto == "Petróleo") & (cuencas.fecha == ult_f)].nlargest(2, "valor")
    if len(cp):
        textos.append(("Cuencas", f"En {mes(ult_f)}, {cp.iloc[0].cuenca} concentró {n(cp.iloc[0].valor/cuencas[(cuencas.producto=='Petróleo') & (cuencas.fecha==ult_f)].valor.sum()*100,0)}% de la producción de petróleo" + (f", seguida por {cp.iloc[1].cuenca}." if len(cp) > 1 else ".")))
resumen = pd.DataFrame(textos, columns=["tema", "texto"])
for t in textos: print("■", t[0] + ":", t[1])

# --- Autenticación: Colab (interactiva) o service account (automática) ---
import gspread
try:
    from google.colab import auth
    auth.authenticate_user()
    from google.auth import default
    creds, _ = default()
except ImportError:          # corre fuera de Colab (ej. GitHub Actions): usa service account
    import os
    from google.oauth2.service_account import Credentials
    creds = Credentials.from_service_account_file(
        os.environ["GOOGLE_SA_JSON"],
        scopes=["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"])
gc = gspread.authorize(creds)
sh = gc.open_by_key(SPREADSHEET_ID)

def a_hoja(nombre, data):
    d_ = data.copy()
    for c in d_.columns:
        if np.issubdtype(d_[c].dtype, np.datetime64): d_[c] = d_[c].dt.strftime("%Y-%m-%d")
        elif np.issubdtype(d_[c].dtype, np.floating): d_[c] = d_[c].round(6)
    d_ = d_.astype(object).where(pd.notna(d_), "")
    vals = [list(d_.columns)] + d_.values.tolist()
    try:
        ws = sh.worksheet(nombre); ws.clear(); ws.resize(rows=max(len(vals), 2), cols=max(len(d_.columns), 1))
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=nombre, rows=len(vals) + 10, cols=len(d_.columns) + 2)
    for i in range(0, len(vals), 20000):
        ws.update(values=vals[i:i + 20000], range_name=f"A{i + 1}", value_input_option="RAW")
    print(f"✔ hoja '{nombre}': {len(d_)} filas")

a_hoja("series", ser)
a_hoja("indicadores", indicadores)
a_hoja("potencia", potencia)
a_hoja("empresas", empresas)
if len(cuencas): a_hoja("cuencas", cuencas)
a_hoja("kpis", kpis)
a_hoja("resumen", resumen)
a_hoja("cobertura", cobertura)
a_hoja("info", pd.DataFrame([{"ultima_actualizacion": pd.Timestamp.now(tz="America/Argentina/Buenos_Aires").strftime("%Y-%m-%d %H:%M"),
                              "fuentes": "CAMMESA, Secretaría de Energía (SESCO y por empresa), ENARGAS e INDEC (vía datos.gob.ar y datos.energia.gob.ar)",
                              "fuentes_usadas": "; ".join(f"{a}: {b}" for a, b in fuentes_usadas) or "las de siempre",
                              "series_atrasadas": "; ".join(atrasadas.id) if len(atrasadas) else "ninguna",
                              "errores": "; ".join(f"{a}: {b}" for a, b in errores) or "ninguno"}]))
for h in ("Hoja 1", "Sheet1"):
    try: sh.del_worksheet(sh.worksheet(h))
    except Exception: pass
print("\nListo ✅  →", f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}")


## Diagnóstico de nuevas fuentes (opcional)

In [ ]:
# =============================================================
# 5) DIAGNÓSTICO (opcional, no escribe nada): buscar series de
#    generación por fuente y precios del MEM para la próxima iteración
# =============================================================
# Correr esta celda y pasarle el resultado a Claude: con los ids que aparezcan
# sumamos "generación mensual por fuente" y "precio monómico" al tablero.
import requests, os
AUTO = bool(os.environ.get("GITHUB_ACTIONS"))  # en la corrida automática se saltea el diagnóstico

def buscar_series(texto, limite=15):
    try:
        j = requests.get("https://apis.datos.gob.ar/series/api/search/",
                         params={"q": texto, "limit": limite}, headers=UA, timeout=60).json()
        for r in j.get("data", []):
            f = r.get("field", {})
            print(f"  {f.get('id','?'):42s} [{r.get('dataset',{}).get('source','')}] {f.get('description','')[:90]}")
    except Exception as e:
        print("  (sin resultados:", e, ")")

for q in [] if AUTO else ["generacion electrica fuente", "generacion termica hidraulica nuclear renovable",
          "precio monomico", "energia electrica generada", "tarifa electricidad"]:
    print(f"\n🔎 '{q}':"); buscar_series(q)

def buscar_ckan(q, limite=10):
    try:
        j = requests.get(f"{CKAN_ENERGIA}/package_search", params={"q": q, "rows": limite}, headers=UA, timeout=60).json()
        for r in j.get("result", {}).get("results", []):
            print(f"  {r.get('name','?'):55s} {r.get('title','')[:70]}")
    except Exception as e:
        print("  (sin resultados:", e, ")")

for q in [] if AUTO else ["generacion", "precio monomico", "cammesa", "ventas combustibles"]:
    print(f"\n🔎 datos.energia.gob.ar '{q}':"); buscar_ckan(q)
